#  如何将Ray集群与Kubernetes集成？

将 Ray 集群与 Kubernetes (K8s) 集成，目前业界最主流、最推荐的方式是使用 KubeRay。

KubeRay 是一个 Kubernetes 操作员（Operator），它允许你通过 Kubernetes 的原生方式（如 YAML 文件、Helm Charts）来定义、部署和管理 Ray 集群。它解决了手动在 K8s 上启动 Ray 的复杂性，实现了“云原生”的 Ray 管理。

以下是将 Ray 与 Kubernetes 集成的详细步骤和架构解析：
## 🏗️ 核心架构：KubeRay
在集成之前，你需要理解 KubeRay 引入的两个核心概念：
1. KubeRay Operator：这是运行在 K8s 集群中的“管家”。它负责监听你发出的指令（比如“创建一个 5 个节点的集群”），并自动调用 K8s API 来创建相应的 Pod 和服务。
2. CRD (自定义资源定义)：KubeRay 扩展了 K8s 的能力，让你可以使用三种新资源：
    - RayCluster：定义一个基础的 Ray 集群（Head + Worker 节点）。
    - RayJob：定义一个一次性任务（如训练任务），任务跑完后集群可以自动销毁。
    - RayService：定义一个长期运行的服务（如模型推理），支持零停机更新。
## 🚀 集成步骤详解
### 第一步：安装 KubeRay Operator

通常使用 Helm 来安装，这是最快捷的方式。
1. 添加 Helm 仓库：

In [ ]:
helm repo add kuberay https://ray-project.github.io/kuberay-helm/
helm repo update

2. 安装 Operator：
- 安装完成后，Operator 会作为一个 Pod 在你的 K8s 集群中运行，随时待命。

In [ ]:
# 在 kube-system 命名空间安装 KubeRay Operator
helm install kuberay-operator kuberay/kuberay-operator \
  --namespace kube-system \
  --version 1.4.2

### 第二步：部署 Ray 集群
安装好 Operator 后，你不再需要手动执行 ray start，而是通过提交 YAML 文件来声明你想要的集群状态。

1. 编写集群配置 (ray-cluster.yaml)
这是一个标准的 RayCluster 定义，指定了 Head 节点和 Worker 节点的资源需求：

In [ ]:
apiVersion: ray.io/v1
kind: RayCluster
metadata:
  name: my-first-ray-cluster
spec:
  # 定义 Head 节点
  headGroupSpec:
    serviceType: ClusterIP
    template:
      spec:
        containers:
        - name: ray-head
          image: rayproject/ray:2.48.0  # 指定 Ray 镜像
          resources:
            limits:
              cpu: "2"
              memory: "4Gi"
            requests:
              cpu: "1"
              memory: "2Gi"
          ports:
          - containerPort: 6379 # GCS 端口
          - containerPort: 8265 # Dashboard 端口
            name: dashboard
  # 定义 Worker 节点组
  workerGroupSpecs:
  - replicas: 2              # 初始 Worker 数量
    minReplicas: 1           # 最小数量（用于自动伸缩）
    maxReplicas: 5           # 最大数量
    groupName: small-workers
    template:
      spec:
        containers:
        - name: ray-worker
          image: rayproject/ray:2.48.0
          resources:
            limits:
              cpu: "4"
              memory: "8Gi"

2. 应用配置
- 此时，KubeRay Operator 会检测到新的 RayCluster 资源，并立即创建 1 个 Head Pod 和 2 个 Worker Pod

In [ ]:
kubectl apply -f ray-cluster.yaml

## 第三步：访问与交互
集群启动后，你可以像操作普通 K8s 资源一样操作它。
- 查看状态：

In [ ]:
kubectl get raycluster
kubectl get pods -l ray.io/cluster=my-first-ray-cluster

- 访问 Dashboard（端口转发）：
    - 然后在浏览器访问 http://localhost:8265

In [ ]:
# 将 Head 节点的 8265 端口转发到本地
kubectl port-forward svc/my-first-ray-cluster-head-svc 8265:8265

- 提交任务：
你可以从本地或 K8s 内部提交任务：

In [ ]:
# 假设你已经配置了 kubectl 连接
ray job submit --address="http://localhost:8265" -- python my_script.py

## 💡 进阶：使用 RayJob 和 RayService
除了基础集群，KubeRay 还提供了更高级的抽象，这才是集成的精髓：
1. 提交训练任务 (RayJob)
如果你只是想跑一个训练脚本，不需要保留集群，使用 RayJob。
- 特点：KubeRay 会自动创建集群 -> 提交任务 -> 监控任务状态 -> 任务完成后自动删除集群。
- 优势：省钱，省心。
2. 部署推理服务 (RayService)
如果你要部署一个在线模型服务。
- 特点：支持零停机更新。当你更新模型版本时，RayService 会先启动新版本的 Worker，等新版本健康检查通过后，再将流量切换过去，最后销毁旧版本。
## 🔑 关键优势总结

| 特性 | 传统 Ray (`ray start`) | Kubernetes + KubeRay |
| :--- | :--- | :--- |
| 资源管理 | 静态，手动指定 IP | 动态，由 K8s 调度器自动分配节点 |
| 扩缩容 | 需手动执行命令 | 支持HPA (水平自动伸缩)，根据 CPU/GPU 利用率自动增减 Pod |
| 高可用 | 需手动配置 | K8s 自动重启失败的 Pod，保证服务在线 |
| 多租户 | 困难 | 利用 K8s Namespace 和 Quota 轻松隔离不同团队 |
| GPU 支持 | 需手动配置驱动 | 利用 K8s Device Plugins 自动透传 GPU 资源 |



## 🛠️ 避坑指南
1. 镜像版本一致性：确保你的 Docker 镜像中包含了所有 Python 依赖（如 pip install ray[default]），且 Ray 版本与 KubeRay Operator 兼容。
2. 资源请求与限制：在 YAML 中务必设置 requests 和 limits。如果不设置，K8s 可能会将 Ray Pod 调度到资源不足的节点上，导致 OOM (内存溢出) 或被 Kill。
3. 网络插件：确保你的 K8s 集群网络插件（CNI）允许 Pod 之间互相通信，Ray 的 GCS 通信对网络延迟比较敏感。
通过 KubeRay，你将 Ray 强大的分布式计算能力与 Kubernetes 成熟的运维体系完美结合，这是目前生产环境的最佳实践。